## OpenAI com skill prompts

Esta secao usa os prompts da skill `ecossistema-agentes-prompts` para orquestrador, especialistas e avaliador.

In [1]:
import os
import re
import json
import sys
import time
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from dotenv import load_dotenv
from mnt.skills.agente_mysql.helpers import MySQLAgent
import logging

load_dotenv()

# Logger do Maestro — troque para logging.DEBUG para ver detalhes de execução
logging.basicConfig(
    level=logging.DEBUG, # WARNING,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s"
)
logger = logging.getLogger("maestro")

sys.path.insert(0, os.path.abspath("."))

In [2]:
# Caminho base das skills (pode ser absoluto ou relativo ao notebook)
SKILLS_DIR = "mnt/skills"
SKILL_FILE = "SKILL.md"

def _parse_frontmatter(content: str) -> Tuple[Dict, str]:
    """Extrai frontmatter YAML (entre ---) e retorna (metadados, resto do conteúdo)."""
    meta = {}
    body = content
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)", content, re.DOTALL)
    if match:
        fm, body = match.group(1), match.group(2)
        for line in fm.split("\n"):
            if ":" in line:
                key, val = line.split(":", 1)
                meta[key.strip()] = val.strip().strip("'\"").replace("\n", " ").strip()
    return meta, body.strip()

def load_skills(base_dir: str) -> List[Dict]:
    """
    Carrega todas as skills a partir de base_dir.
    Cada subpasta contendo SKILL.md vira uma skill.
    Retorna lista de dicts: id, name, description, content (texto completo).
    """
    skills = []
    base = os.path.abspath(base_dir)
    if not os.path.isdir(base):
        return skills
    for entry in sorted(os.listdir(base)):
        path = os.path.join(base, entry)
        skill_file = os.path.join(path, SKILL_FILE)
        if os.path.isdir(path) and os.path.isfile(skill_file):
            with open(skill_file, "r", encoding="utf-8") as f:
                raw = f.read()
            meta, body = _parse_frontmatter(raw)
            skill_id = entry
            name = meta.get("name", skill_id)
            description = meta.get("description", "")
            skills.append({
                "id": skill_id,
                "name": name,
                "description": description,
                "model": meta.get("model"),   # modelo definido no SKILL.md
                "content": raw,
                "body": body,
            })
    return skills

def get_skill_by_id(skills: List[Dict], skill_id: str) -> Optional[Dict]:
    """Retorna a skill com o id dado ou None."""
    for s in skills:
        if s["id"] == skill_id:
            return s
    return None

# Carrega todas as skills
skills_loaded = load_skills(SKILLS_DIR)
print(f"Skills carregadas: {len(skills_loaded)}")
for s in skills_loaded:
    print(f"  - {s['id']}: {s.get('name', s['id'])}")
    # print(s['content'])

Skills carregadas: 11
  - agente-cientifico: agente-cientifico
  - agente-dados: agente-dados
  - agente-financeiro: agente-financeiro
  - agente-juridico: agente-juridico
  - agente-mercado: agente-mercado
  - agente-mysql: agente-mysql
  - agente-negocios: agente-negocios
  - agente-saude: agente-saude
  - agente-tecnico: agente-tecnico
  - avaliador-coerencia: avaliador-coerencia
  - maestro: maestro


In [3]:
## Defini client e o modelo e carregar as skills
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = os.environ.get("MODELO_DEFAULT")
skills = load_skills("mnt/skills")
_t = os.environ.get("TIME_INTERVAL_AGENTS")
time_interval_agents = float(_t) if _t else 2

In [4]:
def chat_with_skill(
    client: OpenAI,
    skill_id: str,
    user_message: str,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Envia uma mensagem ao modelo OpenAI usando o conteúdo da skill como system prompt.
    Usa o modelo definido no SKILL.md (campo model:); fallback para o parâmetro model.
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return f"Erro: skill '{skill_id}' não encontrada."
    modelo = skill.get("model") or model
    resp = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": skill["content"]},
            {"role": "user", "content": user_message},
        ],
    )
    return resp.choices[0].message.content or ""

# Exemplo:
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# model = os.environ.get("MODELO_DEFAULT")
# resposta = chat_with_skill(client, "agente-tecnico", "Como implementar um RAG com embeddings?", model)
# print(resposta)

In [5]:
def extrair_json(texto: str) -> Optional[Dict]:
    """Extrai JSON do texto retornado pelo modelo (pode vir em bloco ```json ... ```)."""
    if not texto or not texto.strip():
        return None
    s = texto.strip()
    # Remove bloco markdown de código
    if "```" in s:
        match = re.search(r"```(?:json)?\s*([\s\S]*?)```", s)
        if match:
            s = match.group(1).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return None

def invocar_agente_maestro(
    client: OpenAI,
    skill_id: str,
    payload_maestro: Dict,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Invoca um agente em modo Maestro; o modelo deve retornar JSON no Formato de Retorno da skill.
    Usa o modelo definido no SKILL.md (campo model:); fallback para o parâmetro model.
    Retorna o texto bruto da resposta (parse com extrair_json depois).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return ""
    modelo = skill.get("model") or model
    user = (
        json.dumps(payload_maestro, ensure_ascii=False)
        + "\n\nVocê está sendo invocado pelo Maestro. Responda APENAS com um único JSON válido no Formato de Retorno da skill "
        "(agente_id, agente_nome, pode_responder, justificativa_viabilidade, resposta, scores, limitacoes_da_resposta, aspectos_para_outros_agentes). "
        "Sem texto antes ou depois."
    )
    resp = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": skill["content"]},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
    )
    return (resp.choices[0].message.content or "").strip()

In [6]:
def executar_fluxo_maestro(
    client: OpenAI,
    pergunta: str,
    model: str = None,
    skills_list: Optional[List[Dict]] = None,
    agentes: Optional[List[str]] = None,
    agentes_dataframe: Optional[List[str]] = None,
    verbose: bool = True,
    mysql_host: Optional[str] = None,
    mysql_porta: Optional[int] = None,
    mysql_usuario: Optional[str] = None,
    mysql_senha: Optional[str] = None,
    mysql_banco: Optional[str] = None,
    mysql_tabela: Optional[str] = None,
    mysql_tabelas: Optional[List[Dict]] = None,
    mysql_limite: int = 50.000,
    mysql_filtro_where: str = "",
    mysql_injetar_namespace: Optional[Dict] = None,
) -> Dict:
    """
    Executa o fluxo completo: análise (1+2), invocação dos N agentes (3),
    payload avaliador (4), avaliador (5), entrega (6).

    Parâmetros novos:
        agentes_dataframe : lista de skill_ids que recebem df_contexto com MySQL.
                            Padrão: ["agente-dados", "agente-financeiro", "agente-negocios"]
        mysql_tabelas     : lista de dicts para múltiplas tabelas com FK.
                            Ex: [{"tabela": "os_servicos", "limite": 50.000},
                                 {"tabela": "servicos", "limite": 5_000,
                                  "fk": "os_servicos.servico_id = servicos.id"}]
                            Tem precedência sobre mysql_tabela quando fornecido.

    Retorna dict com analise, respostas_agentes, avaliacao, entrega_final.
    """
    model = model or os.environ.get("MODELO_DEFAULT")
    skills = skills_list if skills_list is not None else skills_loaded
    agentes = agentes or []
    _agentes_dataframe = agentes_dataframe if agentes_dataframe is not None \
        else ["agente-dados", "agente-financeiro", "agente-negocios"]
    resultado_mysql = None
    df_contexto = None
    df_variavel = None

    def _preparar_contexto_mysql() -> Tuple[Dict, Dict]:
        host    = mysql_host    or os.environ.get("MYSQL_HOST", "localhost")
        porta   = mysql_porta
        if porta is None:
            porta_env = os.environ.get("MYSQL_PORT")
            porta = int(porta_env) if porta_env else 3306
        usuario = mysql_usuario or os.environ.get("MYSQL_USER", "root")
        senha   = mysql_senha   or os.environ.get("MYSQL_PASSWORD", "")
        banco   = mysql_banco   or os.environ.get("MYSQL_DATABASE", "")

        agent     = MySQLAgent(host=host, porta=porta, usuario=usuario, senha=senha, banco=banco)
        namespace = mysql_injetar_namespace if mysql_injetar_namespace is not None else globals()

        # ── Modo múltiplas tabelas via LEFT JOIN ──────────────────────────
        if mysql_tabelas:
            resultado = agent.carregar_multiplas_tabelas(
                definicoes=mysql_tabelas,
                limite=mysql_limite,
                filtro_where=mysql_filtro_where,
                verbose=verbose,
            )
        else:
            # ── Modo tabela única (legado) ────────────────────────────────
            resultado = agent.carregar_tabela(
                mysql_tabela,
                limite=mysql_limite,
                filtro_where=mysql_filtro_where,
                verbose=verbose,
            )

        if not resultado["sucesso"]:
            raise RuntimeError(f"[agente-mysql] {resultado['erro']}")

        if namespace is not None:
            agent.injetar_no_namespace(resultado, namespace)

        df = resultado["dataframe"]
        contexto = {
            "df_variavel": resultado["variavel"],
            "df_info":     resultado["metadados"]["df_info"],
            "df_colunas":  resultado["metadados"]["colunas"],
            "df_amostra":  df.head(10).to_json(orient="records", force_ascii=False),
        }
        return resultado, contexto

    def _executar_codigo_agente(parsed: Dict) -> Dict:
        codigo = parsed.get("codigo_pandas")
        if not codigo:
            return {"sucesso": False, "resultado": ""}

        df_nome    = parsed.get("df_variavel_usada") or (df_contexto or {}).get("df_variavel")
        namespace  = mysql_injetar_namespace if mysql_injetar_namespace is not None else globals()
        exec_globals = namespace if namespace is not None else globals()
        if "__builtins__" not in exec_globals:
            exec_globals["__builtins__"] = __builtins__

        if df_nome and df_nome not in exec_globals:
            msg = f"DataFrame '{df_nome}' não encontrado no namespace."
            logger.warning(msg)
            return {"sucesso": False, "resultado": msg}

        import io, contextlib
        exec_ns = dict(exec_globals)
        exec_ns.pop("resultado", None)

        logger.debug(json.dumps({
            "location": "executar_fluxo:pre_exec",
            "agente": parsed.get("agente_id"),
            "df_variavel_usada": df_nome,
            "df_ctx_variavel": (df_contexto or {}).get("df_variavel"),
            "codigo_preview": codigo[:800],
        }, ensure_ascii=False))

        try:
            stdout = io.StringIO()
            with contextlib.redirect_stdout(stdout):
                exec(codigo, exec_ns)

            resultado_obj = exec_ns.get("resultado")
            if resultado_obj is None:
                resultado_texto = stdout.getvalue().strip()
            else:
                try:
                    resultado_texto = resultado_obj.to_string() \
                        if hasattr(resultado_obj, "to_string") else str(resultado_obj)
                except Exception:
                    resultado_texto = str(resultado_obj)

            if not resultado_texto:
                resultado_texto = stdout.getvalue().strip()
            if not resultado_texto:
                resultado_texto = "(sem saída)"

            logger.debug(json.dumps({
                "location": "executar_fluxo:post_exec",
                "agente": parsed.get("agente_id"),
                "sucesso": True,
                "resultado_preview": resultado_texto[:500],
            }, ensure_ascii=False))

            return {"sucesso": True, "resultado": resultado_texto}

        except Exception as exc:
            logger.warning(f"Erro ao executar codigo_pandas do agente {parsed.get('agente_id')}: {exc}")
            return {"sucesso": False, "resultado": f"Erro ao executar codigo_pandas: {exc}"}

    if verbose:
        print("[MAESTRO] Iniciando fluxo. Pergunta:", (pergunta[:80] + "...") if len(pergunta) > 80 else pergunta)

    # --- Passo 1+2: Análise da pergunta ---
    if verbose:
        print("[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...")
    skill_maestro = get_skill_by_id(skills, "maestro")
    model_maestro = skill_maestro.get("model") or model
    msg_analise = (
        pergunta
        + "\n\nRetorne um JSON com uma única chave 'analise' (objeto) contendo: "
        "pergunta (string), dominios_identificados (lista de strings), tipo_resposta_esperada (uma de: factual, analítica, técnica, criativa, comparativa), "
        "complexidade (uma de: baixa, média, alta), informacao_central (string)."
    )
    resp_analise = client.chat.completions.create(
        model=model_maestro,
        messages=[
            {"role": "system", "content": skill_maestro["content"]},
            {"role": "user",   "content": msg_analise},
        ],
        response_format={"type": "json_object"},
    )
    raw_analise  = (resp_analise.choices[0].message.content or "").strip()
    analise_data = extrair_json(raw_analise)
    analise = (analise_data.get("analise") or analise_data) if isinstance(analise_data, dict) else {}
    if not analise:
        analise = {
            "pergunta": pergunta,
            "dominios_identificados": [],
            "tipo_resposta_esperada": "analítica",
            "complexidade": "média",
            "informacao_central": pergunta,
        }
    contexto_maestro = (
        f"Domínios: {analise.get('dominios_identificados', [])}; "
        f"Tipo: {analise.get('tipo_resposta_esperada', '')}; "
        f"Complexidade: {analise.get('complexidade', '')}; "
        f"Central: {analise.get('informacao_central', '')}"
    )
    if verbose:
        print("[MAESTRO] Análise obtida:", analise.get("dominios_identificados"),
              analise.get("tipo_resposta_esperada"), analise.get("complexidade"))

    if mysql_tabelas or mysql_tabela:
        if verbose:
            print("[MAESTRO] Pré-carregando dados MySQL...")
        resultado_mysql, df_contexto = _preparar_contexto_mysql()
        df_variavel = df_contexto.get("df_variavel")
        if verbose:
            print(f"[MAESTRO] DataFrame disponível em '{df_variavel}'.")
        tabela_ref = (mysql_tabelas[0]["tabela"] if mysql_tabelas else mysql_tabela)
        contexto_maestro = f"{contexto_maestro}; Tabela: {tabela_ref}; DF: {df_variavel}"

    # --- Passo 3: Invocar os agentes (fluxo 2 fases para agentes DataFrame) ---
    respostas_agentes = []
    for skill_id in agentes:
        usa_dataframe = bool(df_contexto and skill_id in _agentes_dataframe)

        if verbose:
            modo = "2 fases" if usa_dataframe else "conhecimento"
            print(f"[MAESTRO] Passo 3: Invocando agente: {skill_id} ({modo})")

        # ── Payload base ──────────────────────────────────────────────────────
        payload_base = {
            "skill_invocada": skill_id,
            "pergunta":       pergunta,
            "contexto_maestro": contexto_maestro,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
        }

        # ── Agentes sem DataFrame: fluxo original ─────────────────────────────
        if not usa_dataframe:
            payload_base["instrucao"] = "Responda estritamente dentro do seu domínio. Calcule seus scores."
            raw    = invocar_agente_maestro(client, skill_id, payload_base, model=model, skills_list=skills)
            parsed = extrair_json(raw)
            if parsed and isinstance(parsed, dict):
                parsed.setdefault("agente_id", skill_id)
                respostas_agentes.append(parsed)
                if verbose:
                    print("[MAESTRO]   ->", skill_id,
                          "pode_responder:", parsed.get("pode_responder"),
                          "score_final:", (parsed.get("scores") or {}).get("score_final"))
            else:
                respostas_agentes.append({
                    "agente_id": skill_id, "agente_nome": skill_id,
                    "pode_responder": False,
                    "justificativa_viabilidade": "Resposta não veio em JSON válido.",
                    "resposta": "", "scores": {},
                })
                if verbose:
                    print("[MAESTRO]   ->", skill_id, "pode_responder: False (parse falhou)")
            continue

        # ── Agentes DataFrame: FASE 1 — extração ─────────────────────────────
        payload_fase1 = {**payload_base}
        payload_fase1.update(df_contexto)
        payload_fase1["fase"]     = "extracao"
        payload_fase1["instrucao"] = (
            "FASE 1 — EXTRAÇÃO: opere em MODO DATAFRAME. "
            "Gere codigo_pandas específico para o seu domínio conforme as instruções da sua skill. "
            "Retorne o JSON com codigo_pandas preenchido."
        )

        if verbose:
            print(f"[MAESTRO]   [{skill_id}] FASE 1: solicitando código de extração...")

        raw1    = invocar_agente_maestro(client, skill_id, payload_fase1, model=model, skills_list=skills)
        parsed1 = extrair_json(raw1)

        if not parsed1 or not isinstance(parsed1, dict):
            if verbose:
                print(f"[MAESTRO]   [{skill_id}] FASE 1 falhou: parse JSON inválido")
            respostas_agentes.append({
                "agente_id": skill_id, "agente_nome": skill_id,
                "pode_responder": False,
                "justificativa_viabilidade": "FASE 1 não retornou JSON válido.",
                "resposta": "", "scores": {},
            })
            continue

        parsed1.setdefault("agente_id", skill_id)

        if not parsed1.get("codigo_pandas"):
            if verbose:
                print(f"[MAESTRO]   [{skill_id}] FASE 1: agente optou por não gerar código")
            # Agente decidiu que não pode responder — mantém a resposta da fase 1
            respostas_agentes.append(parsed1)
            if verbose:
                print("[MAESTRO]   ->", skill_id,
                      "pode_responder:", parsed1.get("pode_responder"),
                      "score_final:", (parsed1.get("scores") or {}).get("score_final"))
            continue

        # Executa o código gerado na FASE 1
        exec_info = _executar_codigo_agente(parsed1)
        resultado_extracao = exec_info.get("resultado", "")

        if verbose:
            status_exec = "✅" if exec_info.get("sucesso") else "❌"
            preview = (resultado_extracao[:120] + "...") if len(resultado_extracao) > 120 else resultado_extracao
            print(f"[MAESTRO]   [{skill_id}] FASE 1 execução {status_exec}: {preview!r}")

        if not exec_info.get("sucesso") or not resultado_extracao:
            if verbose:
                print(f"[MAESTRO]   [{skill_id}] FASE 1 execução falhou — pulando FASE 2")
            parsed1["resposta"] = (
                (parsed1.get("resposta") or "") +
                f"\n\nErro na execução do código:\n{resultado_extracao}"
            ).strip()
            parsed1["resultado_execucao"] = resultado_extracao
            respostas_agentes.append(parsed1)
            if verbose:
                print("[MAESTRO]   ->", skill_id,
                      "pode_responder:", parsed1.get("pode_responder"),
                      "score_final:", (parsed1.get("scores") or {}).get("score_final"))
            continue

        # ── FASE 2 — interpretação ────────────────────────────────────────────
        if verbose:
            print(f"[MAESTRO]   [{skill_id}] FASE 2: enviando dados reais para interpretação...")

        payload_fase2 = {
            "skill_invocada":   skill_id,
            "pergunta":         pergunta,
            "contexto_maestro": contexto_maestro,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "fase":             "interpretacao",
            "resultado_extracao": resultado_extracao,
            "instrucao": (
                "FASE 2 — INTERPRETAÇÃO: você recebeu em 'resultado_extracao' os dados reais "
                "extraídos do banco de dados. Interprete esses dados conforme o seu domínio e "
                "responda à pergunta do usuário com análise fundamentada nos números reais. "
                "NÃO repita os dados brutos — sintetize em insights. "
                "Retorne o JSON completo de resposta (sem codigo_pandas)."
            ),
        }

        raw2    = invocar_agente_maestro(client, skill_id, payload_fase2, model=model, skills_list=skills)
        parsed2 = extrair_json(raw2)

        if parsed2 and isinstance(parsed2, dict):
            parsed2.setdefault("agente_id", skill_id)
            parsed2["resultado_execucao"] = resultado_extracao  # preserva para debug
            respostas_agentes.append(parsed2)
            if verbose:
                print("[MAESTRO]   ->", skill_id,
                      "pode_responder:", parsed2.get("pode_responder"),
                      "score_final:", (parsed2.get("scores") or {}).get("score_final"))
        else:
            if verbose:
                print(f"[MAESTRO]   [{skill_id}] FASE 2 falhou: usando resposta da FASE 1")
            # Fallback: usa a resposta da fase 1 com o resultado de execução
            parsed1["resultado_execucao"] = resultado_extracao
            parsed1["resposta"] = (
                (parsed1.get("resposta") or "") + f"\n\nResultado:\n{resultado_extracao}"
            ).strip()
            respostas_agentes.append(parsed1)
            if verbose:
                print("[MAESTRO]   ->", skill_id,
                      "pode_responder:", parsed1.get("pode_responder"),
                      "score_final:", (parsed1.get("scores") or {}).get("score_final"))

    # --- Passo 4: Montar payload do avaliador ---
    para_avaliador = [r for r in respostas_agentes if r.get("pode_responder") is True]
    if verbose:
        print("[MAESTRO] Passo 4: Respostas a enviar ao avaliador:",
              len(para_avaliador), "de", len(respostas_agentes))

    avaliacao = None
    if not para_avaliador:
        entrega_final = (
            "## Resposta do Maestro\n\n**Pergunta:** " + pergunta
            + "\n\nNenhum agente qualificado para responder. "
            "Sugestão: reformule a pergunta ou consulte um domínio mais específico."
        )
    else:
        # --- Passo 5: Invocar avaliador de coerência ---
        if verbose:
            print("[MAESTRO] Passo 5: Invocando avaliador de coerência...")
        payload_aval = {
            "pergunta_original": pergunta,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "respostas_coletadas": [
                {
                    "agente_id":   r.get("agente_id"),
                    "agente_nome": r.get("agente_nome", r.get("agente_id")),
                    "resposta":    r.get("resposta", ""),
                    "scores_agente": r.get("scores") or {},
                    "limitacoes_da_resposta": r.get("limitacoes_da_resposta", ""),
                }
                for r in para_avaliador
            ],
        }
        skill_aval = get_skill_by_id(skills, "avaliador-coerencia")
        model_aval = skill_aval.get("model") or model
        user_aval  = (
            json.dumps(payload_aval, ensure_ascii=False)
            + "\n\nRetorne APENAS um JSON com: avaliacao_completa (lista de objetos com "
            "agente_id, agente_nome, scores_avaliador, score_total, status, observacoes), "
            "ranking_final (lista de agente_id), conflitos_detectados, respostas_excluidas, threshold_utilizado."
        )
        resp_aval = client.chat.completions.create(
            model=model_aval,
            messages=[
                {"role": "system", "content": skill_aval["content"]},
                {"role": "user",   "content": user_aval},
            ],
            response_format={"type": "json_object"},
        )
        raw_aval  = (resp_aval.choices[0].message.content or "").strip()
        avaliacao = extrair_json(raw_aval)
        if verbose:
            ranking = (avaliacao or {}).get("ranking_final", [])
            print("[MAESTRO] Avaliador retornou ranking:", ranking)
            if (avaliacao or {}).get("conflitos_detectados"):
                print("[MAESTRO]   Conflitos:", avaliacao.get("conflitos_detectados"))

        # --- Passo 6: Entrega ao usuário ---
        if verbose:
            print("[MAESTRO] Passo 6: Formatando entrega ao usuário.")
        entrega_final = _formatar_entrega(pergunta, respostas_agentes, avaliacao, para_avaliador)

    if verbose:
        print("[MAESTRO] Fluxo concluído.")
    return {
        "analise":          analise,
        "respostas_agentes": respostas_agentes,
        "avaliacao":        avaliacao,
        "entrega_final":    entrega_final,
        "resultado_mysql":  resultado_mysql,
        "df_variavel":      df_variavel,
    }


def _formatar_entrega(pergunta: str, respostas_agentes: List[Dict], avaliacao: Optional[Dict], para_avaliador: List[Dict]) -> str:
    """Gera o markdown da entrega conforme PASSO 6 do Maestro."""
    def _resposta_para_str(resp):
        if isinstance(resp, dict):
            return resp.get("resumo_executivo", json.dumps(resp, ensure_ascii=False))
        return resp if isinstance(resp, str) else str(resp)
    n_consultados  = len(respostas_agentes)
    aval           = avaliacao or {}
    avaliacao_completa = aval.get("avaliacao_completa") or []
    ranking        = aval.get("ranking_final") or [r.get("agente_id") for r in para_avaliador]
    id_to_resp     = {r.get("agente_id"): r for r in para_avaliador}
    ordenados      = [id_to_resp[aid] for aid in ranking if aid in id_to_resp]
    for r in para_avaliador:
        if r.get("agente_id") not in ranking:
            ordenados.append(r)
    n_qualificadas = len(ordenados)
    linhas = [
        "## Resposta do Maestro", "",
        "**Pergunta:** " + pergunta,
        f"**Agentes consultados:** {n_consultados} | **Respostas qualificadas:** {n_qualificadas}",
        "---",
    ]
    for r in ordenados:
        aid        = r.get("agente_id", "")
        nome       = r.get("agente_nome", aid)
        score_total = next(
            (item.get("score_total") for item in avaliacao_completa if item.get("agente_id") == aid),
            (r.get("scores") or {}).get("score_final", 0)
        )
        pct = round((score_total or 0) * 100)
        linhas += [f"### {nome} — {aid}", f"*Score de Confiança: {pct}%*", "", _resposta_para_str(r.get("resposta", "")), "", "---"]
    linhas += ["", "### Síntese"]
    resumos = []
    for r in ordenados:
        s = _resposta_para_str(r.get("resposta", ""))
        resumos.append((s[:120] + "...") if len(s) > 120 else s)
    linhas.append("Integração das perspectivas: " + " | ".join(resumos))
    return "\n".join(linhas)


In [7]:
# Fluxo seguro: FASE 1 com perguntas agregadas (JSON) + amostra sanitizada

def invocar_agente_maestro(
    client: OpenAI,
    skill_id: str,
    payload_maestro: Dict,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Invoca um agente em modo Maestro e exige retorno JSON válido.
    O contrato base permanece o mesmo, com possibilidade de campos extras por fase
    (ex.: perguntas_dados na FASE 1 de agentes DataFrame).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return ""
    modelo = skill.get("model") or model
    user = (
        json.dumps(payload_maestro, ensure_ascii=False)
        + "\n\nVocê está sendo invocado pelo Maestro. Responda APENAS com um único JSON válido no Formato de Retorno da skill "
        "(agente_id, agente_nome, pode_responder, justificativa_viabilidade, resposta, scores, limitacoes_da_resposta, aspectos_para_outros_agentes), "
        "incluindo campos extras de fase quando solicitado (ex.: perguntas_dados na FASE 1). "
        "Sem texto antes ou depois."
    )
    resp = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": skill["content"]},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
    )
    return (resp.choices[0].message.content or "").strip()


def executar_fluxo_maestro(
    client: OpenAI,
    pergunta: str,
    model: str = None,
    skills_list: Optional[List[Dict]] = None,
    agentes: Optional[List[str]] = None,
    agentes_dataframe: Optional[List[str]] = None,
    verbose: bool = True,
    mysql_host: Optional[str] = None,
    mysql_porta: Optional[int] = None,
    mysql_usuario: Optional[str] = None,
    mysql_senha: Optional[str] = None,
    mysql_banco: Optional[str] = None,
    mysql_tabela: Optional[str] = None,
    mysql_tabelas: Optional[List[Dict]] = None,
    mysql_limite: int = 50.000,
    mysql_filtro_where: str = "",
    mysql_injetar_namespace: Optional[Dict] = None,
) -> Dict:
    """
    Executa o fluxo completo: análise (1+2), invocação dos N agentes (3),
    payload avaliador (4), avaliador (5), entrega (6).

    Versão com privacidade reforçada para agentes DataFrame:
      - FASE 1: agente retorna perguntas agregadas em JSON (perguntas_dados)
      - Executor interno calcula somente métricas agregadas (sem linha crua)
      - FASE 2: agente interpreta resultado_extracao estruturado
    """
    import pandas as pd

    model = model or os.environ.get("MODELO_DEFAULT")
    skills = skills_list if skills_list is not None else skills_loaded
    agentes = agentes or []
    _agentes_dataframe = agentes_dataframe if agentes_dataframe is not None \
        else ["agente-dados", "agente-financeiro", "agente-negocios"]
    resultado_mysql = None
    df_contexto = None
    df_variavel = None

    contrato_perguntas = {
        "schema": "perguntas_dados.v1",
        "campos": [
            "metric_id", "descricao", "tipo", "coluna_valor", "group_by", "filtros", "janela_tempo", "top_n",
            "coluna_data", "frequencia", "quantil", "agregacao"
        ],
        "tipos_permitidos": ["count", "sum", "mean", "median", "percentile", "top_n", "timeseries", "null_rate", "nunique"],
        "operadores_filtro_permitidos": ["eq", "ne", "gt", "gte", "lt", "lte", "in", "not_in"],
    }

    def _to_scalar(v):
        try:
            if pd.isna(v):
                return None
        except Exception:
            pass
        if hasattr(v, "item"):
            try:
                return v.item()
            except Exception:
                return str(v)
        return v

    def _resolver_col_data(df, preferida=None):
        candidatos = []
        if preferida:
            candidatos.append(preferida)
        candidatos += ["created_at", "data", "data_venda", "updated_at", "timestamp"]
        for c in candidatos:
            if c in df.columns:
                return c
        return None

    def _mascarar_valor(valor):
        if valor is None:
            return None
        if isinstance(valor, str):
            return (valor[:64] + "...") if len(valor) > 64 else valor
        return valor

    def _sanitizar_df_amostra(df: pd.DataFrame, n: int = 8) -> List[Dict]:
        if df is None or df.empty:
            return []

        import re as _re
        import numpy as _np
        import pandas as _pd

        sensivel_regex = _re.compile(
            r"(email|mail|telefone|celular|cpf|cnpj|document|rg|endereco|address|senha|password|token|secret|pix|cartao|card|nascimento)",
            _re.IGNORECASE,
        )
        id_regex = _re.compile(r"(^id$|_id$|id_|uuid)", _re.IGNORECASE)
        amostra = df.head(n).copy()
        total = len(df)

        for i in range(len(amostra.columns)):
            col = amostra.columns[i]
            serie = amostra.iloc[:, i]
            col_s = str(col)
            col_low = col_s.lower()
            if sensivel_regex.search(col_low):
                amostra.iloc[:, i] = amostra.iloc[:, i].astype(object)
                amostra.iloc[:, i] = "***MASKED***"
                continue

            try:
                nunique = int(df.iloc[:, i].nunique(dropna=True))
            except Exception:
                nunique = None

            if id_regex.search(col_low) and nunique is not None and total > 0 and (nunique / total) > 0.8:
                amostra.iloc[:, i] = amostra.iloc[:, i].astype(object)
                amostra.iloc[:, i] = "<ID_MASKED>"
                continue

            if serie.dtype == "object" or str(serie.dtype) == "object":
                amostra.iloc[:, i] = serie.map(_mascarar_valor)

        if amostra.columns.duplicated().any():
            seen = {}
            new_cols = []
            for c in amostra.columns:
                cstr = str(c)
                n = seen.get(cstr, 0)
                new_cols.append(cstr if n == 0 else f"{cstr}_{n}")
                seen[cstr] = n + 1
            amostra.columns = new_cols
        records = amostra.to_dict(orient="records")

        def _to_json_serializable(val):
            # None or NaN/NaT -> None
            try:
                if pd.isna(val):
                    return None
            except Exception:
                pass
            # pandas Timestamp
            try:
                if isinstance(val, _pd.Timestamp):
                    return val.isoformat()
            except Exception:
                pass
            # numpy scalar types
            try:
                if isinstance(val, _np.generic):
                    return val.item()
            except Exception:
                pass
            # datetimes
            try:
                import datetime as _dt
                if isinstance(val, _dt.datetime) or isinstance(val, _dt.date):
                    return val.isoformat()
            except Exception:
                pass
            # timedelta
            try:
                if isinstance(val, _pd.Timedelta):
                    return str(val)
            except Exception:
                pass
            # fallback for common primitives
            if isinstance(val, (str, int, float, bool)):
                return val
            # default to string representation
            try:
                return str(val)
            except Exception:
                return None

        clean_records = []
        for r in records:
            nr = {}
            for k, v in r.items():
                nr[k] = _to_json_serializable(v)
            clean_records.append(nr)

        return clean_records

    def _montar_df_perfil(df: pd.DataFrame) -> Dict:
        if df is None:
            return {}

        total = len(df)
        cols = []
        n_cols = min(80, len(df.columns))
        for i in range(n_cols):
            serie = df.iloc[:, i]
            col = df.columns[i]
            nulls = int(serie.isna().sum())
            pct_null = round((nulls / total) * 100, 2) if total > 0 else 0.0
            try:
                nunique = int(serie.nunique(dropna=True))
            except Exception:
                nunique = None
            cols.append({
                "coluna": str(col),
                "dtype": str(serie.dtype),
                "nulls": nulls,
                "pct_null": pct_null,
                "nunique": nunique,
            })

        num_indices = [i for i in range(len(df.columns)) if pd.api.types.is_numeric_dtype(df.iloc[:, i].dtype)][:15]
        estat_num = {}
        seen_num = {}
        for i in num_indices:
            s = pd.to_numeric(df.iloc[:, i], errors="coerce")
            c = str(df.columns[i])
            key = c if c not in seen_num else f"{c}_{seen_num[c]}"
            seen_num[c] = seen_num.get(c, 0) + 1
            estat_num[key] = {
                "mean": _to_scalar(round(s.mean(), 6)) if s.notna().any() else None,
                "median": _to_scalar(round(s.median(), 6)) if s.notna().any() else None,
                "p95": _to_scalar(round(s.quantile(0.95), 6)) if s.notna().any() else None,
                "min": _to_scalar(round(s.min(), 6)) if s.notna().any() else None,
                "max": _to_scalar(round(s.max(), 6)) if s.notna().any() else None,
            }

        return {
            "linhas": int(df.shape[0]),
            "colunas": int(df.shape[1]),
            "perfil_colunas": cols,
            "estatisticas_numericas": estat_num,
        }

    def _preparar_contexto_mysql() -> Tuple[Dict, Dict]:
        host    = mysql_host    or os.environ.get("MYSQL_HOST", "localhost")
        porta   = mysql_porta
        if porta is None:
            porta_env = os.environ.get("MYSQL_PORT")
            porta = int(porta_env) if porta_env else 3306
        usuario = mysql_usuario or os.environ.get("MYSQL_USER", "root")
        senha   = mysql_senha   or os.environ.get("MYSQL_PASSWORD", "")
        banco   = mysql_banco   or os.environ.get("MYSQL_DATABASE", "")

        agent     = MySQLAgent(host=host, porta=porta, usuario=usuario, senha=senha, banco=banco)
        namespace = mysql_injetar_namespace if mysql_injetar_namespace is not None else globals()

        if mysql_tabelas:
            resultado = agent.carregar_multiplas_tabelas(
                definicoes=mysql_tabelas,
                limite=mysql_limite,
                filtro_where=mysql_filtro_where,
                verbose=verbose,
            )
        else:
            resultado = agent.carregar_tabela(
                mysql_tabela,
                limite=mysql_limite,
                filtro_where=mysql_filtro_where,
                verbose=verbose,
            )

        if not resultado["sucesso"]:
            raise RuntimeError(f"[agente-mysql] {resultado['erro']}")

        if namespace is not None:
            agent.injetar_no_namespace(resultado, namespace)

        df = resultado["dataframe"]
        amostra_sanitizada = _sanitizar_df_amostra(df)
        # Instrumentation: write a small summary to the debug log before serializing
        import time
        try:
            summary = {
                "sessionId": "f97b50",
                "id": f"log_{int(time.time()*1000)}",
                "timestamp": int(time.time()*1000),
                "location": "agentes_simples_4.ipynb:_preparar_contexto_mysql",
                "message": "amostra_sanitizada_summary",
                "runId": "pre_fix",
                "hypothesisId": "A",
                "data": {
                    "n_rows": len(amostra_sanitizada),
                    "sample_columns": list(amostra_sanitizada[0].keys()) if amostra_sanitizada else []
                }
            }
        except Exception:
            summary = {"sessionId":"f97b50","id":f"log_{int(time.time()*1000)}","timestamp":int(time.time()*1000),"location":"agentes_simples_4.ipynb:_preparar_contexto_mysql","message":"amostra_summary_error","runId":"pre_fix","hypothesisId":"A","data":{}}
        try:
            with open("/home/cristiano/code/jupter_notebook/.cursor/debug-f97b50.log", "a", encoding="utf-8") as _lf:
                _lf.write(json.dumps(summary, ensure_ascii=False) + "\n")
        except Exception:
            pass

        # Try to serialize to detect problematic types; log and re-raise on failure to collect evidence
        try:
            amostra_json = json.dumps(amostra_sanitizada, ensure_ascii=False)
            amostra_sanitizada_json = amostra_json
        except TypeError as exc:
            errlog = {
                "sessionId": "f97b50",
                "id": f"log_{int(time.time()*1000)}",
                "timestamp": int(time.time()*1000),
                "location": "agentes_simples_4.ipynb:_preparar_contexto_mysql",
                "message": "json_dump_error",
                "runId": "pre_fix",
                "hypothesisId": "A",
                "data": {"error": str(exc)}
            }
            try:
                with open("/home/cristiano/code/jupter_notebook/.cursor/debug-f97b50.log", "a", encoding="utf-8") as _lf:
                    _lf.write(json.dumps(errlog, ensure_ascii=False) + "\n")
            except Exception:
                pass
            raise

        contexto = {
            "df_variavel": resultado["variavel"],
            "df_info": resultado["metadados"]["df_info"],
            "df_colunas": resultado["metadados"]["colunas"],
            "df_perfil": _montar_df_perfil(df),
            "df_amostra": amostra_sanitizada_json,
            "df_amostra_sanitizada": amostra_sanitizada_json,
            "contrato_perguntas_dados": contrato_perguntas,
        }
        return resultado, contexto

    def _aplicar_filtros(df: pd.DataFrame, filtros: List[Dict]) -> pd.DataFrame:
        out = df
        for f in filtros or []:
            col = f.get("coluna")
            op = (f.get("operador") or "eq").lower()
            val = f.get("valor")
            if not col or col not in out.columns:
                raise ValueError(f"Filtro inválido: coluna '{col}' não existe")
            if op == "eq":
                out = out[out[col] == val]
            elif op == "ne":
                out = out[out[col] != val]
            elif op == "gt":
                out = out[out[col] > val]
            elif op == "gte":
                out = out[out[col] >= val]
            elif op == "lt":
                out = out[out[col] < val]
            elif op == "lte":
                out = out[out[col] <= val]
            elif op == "in":
                vals = val if isinstance(val, list) else [val]
                out = out[out[col].isin(vals)]
            elif op == "not_in":
                vals = val if isinstance(val, list) else [val]
                out = out[~out[col].isin(vals)]
            else:
                raise ValueError(f"Operador de filtro não permitido: {op}")
        return out

    def _aplicar_janela(df: pd.DataFrame, item: Dict) -> pd.DataFrame:
        janela = item.get("janela_tempo")
        if not janela:
            return df

        dias = None
        col_data = item.get("coluna_data")
        if isinstance(janela, dict):
            dias = janela.get("dias")
            col_data = janela.get("coluna") or col_data
        elif isinstance(janela, (int, float)):
            dias = int(janela)

        if not dias:
            return df

        col_data = _resolver_col_data(df, preferida=col_data)
        if not col_data:
            raise ValueError("Janela temporal solicitada, mas coluna de data não encontrada")

        dt = pd.to_datetime(df[col_data], errors="coerce")
        limite = pd.Timestamp.now() - pd.Timedelta(days=int(dias))
        return df[dt >= limite]

    def _executar_perguntas_agregadas(parsed: Dict) -> Dict:
        perguntas = parsed.get("perguntas_dados") or []
        if not isinstance(perguntas, list) or len(perguntas) == 0:
            return {
                "sucesso": False,
                "resultado_texto": "perguntas_dados ausente ou vazio",
                "resultado_obj": {"schema_version": "1.0", "metricas": [], "erros": ["perguntas_dados ausente ou vazio"]},
            }

        df_nome = parsed.get("df_variavel_usada") or (df_contexto or {}).get("df_variavel")
        namespace = mysql_injetar_namespace if mysql_injetar_namespace is not None else globals()
        if not df_nome or df_nome not in namespace:
            msg = f"DataFrame '{df_nome}' não encontrado no namespace"
            return {
                "sucesso": False,
                "resultado_texto": msg,
                "resultado_obj": {"schema_version": "1.0", "metricas": [], "erros": [msg]},
            }

        base_df = namespace[df_nome]
        metricas = []
        erros = []

        for idx, item in enumerate(perguntas):
            metric_id = item.get("metric_id") or f"m_{idx+1}"
            desc = item.get("descricao") or ""
            tipo = (item.get("tipo") or "").lower().strip()
            try:
                if tipo not in contrato_perguntas["tipos_permitidos"]:
                    raise ValueError(f"Tipo não permitido: {tipo}")

                trabalho = base_df.copy()
                trabalho = _aplicar_filtros(trabalho, item.get("filtros") or [])
                trabalho = _aplicar_janela(trabalho, item)

                group_by = item.get("group_by") or []
                if isinstance(group_by, str):
                    group_by = [group_by]
                for g in group_by:
                    if g not in trabalho.columns:
                        raise ValueError(f"group_by inválido: coluna '{g}' não existe")

                coluna_valor = item.get("coluna_valor")

                if tipo == "count":
                    if coluna_valor:
                        if coluna_valor not in trabalho.columns:
                            raise ValueError(f"coluna_valor inválida: {coluna_valor}")
                        valor = int(trabalho[coluna_valor].notna().sum())
                    else:
                        valor = int(len(trabalho))

                elif tipo in {"sum", "mean", "median"}:
                    if not coluna_valor or coluna_valor not in trabalho.columns:
                        raise ValueError("sum/mean/median exigem coluna_valor válida")
                    s = pd.to_numeric(trabalho[coluna_valor], errors="coerce")
                    if tipo == "sum":
                        valor = _to_scalar(round(s.sum(), 6))
                    elif tipo == "mean":
                        valor = _to_scalar(round(s.mean(), 6)) if s.notna().any() else None
                    else:
                        valor = _to_scalar(round(s.median(), 6)) if s.notna().any() else None

                elif tipo == "percentile":
                    if not coluna_valor or coluna_valor not in trabalho.columns:
                        raise ValueError("percentile exige coluna_valor válida")
                    q = item.get("quantil")
                    if q is None:
                        q = item.get("percentil")
                    q = float(q if q is not None else 0.95)
                    if q > 1:
                        q = q / 100.0
                    q = min(max(q, 0.0), 1.0)
                    s = pd.to_numeric(trabalho[coluna_valor], errors="coerce")
                    valor = _to_scalar(round(s.quantile(q), 6)) if s.notna().any() else None

                elif tipo == "null_rate":
                    col = coluna_valor or item.get("coluna")
                    if not col or col not in trabalho.columns:
                        raise ValueError("null_rate exige coluna válida")
                    valor = round(float(trabalho[col].isna().mean() * 100), 4)

                elif tipo == "nunique":
                    col = coluna_valor or item.get("coluna")
                    if not col or col not in trabalho.columns:
                        raise ValueError("nunique exige coluna válida")
                    valor = int(trabalho[col].nunique(dropna=True))

                elif tipo == "top_n":
                    n = int(item.get("top_n") or 10)
                    agg = (item.get("agregacao") or "count").lower()
                    grp = group_by[0] if group_by else item.get("coluna_grupo")
                    if not grp or grp not in trabalho.columns:
                        raise ValueError("top_n exige group_by ou coluna_grupo válida")

                    if agg == "count":
                        serie = trabalho[grp].value_counts(dropna=False).head(n)
                        valor = [{"grupo": _to_scalar(k), "valor": int(v)} for k, v in serie.items()]
                    else:
                        if not coluna_valor or coluna_valor not in trabalho.columns:
                            raise ValueError("top_n com agregacao != count exige coluna_valor")
                        s = pd.to_numeric(trabalho[coluna_valor], errors="coerce")
                        df_aux = trabalho.copy()
                        df_aux["__valor__"] = s
                        if agg == "sum":
                            serie = df_aux.groupby(grp, dropna=False)["__valor__"].sum().sort_values(ascending=False).head(n)
                        elif agg == "mean":
                            serie = df_aux.groupby(grp, dropna=False)["__valor__"].mean().sort_values(ascending=False).head(n)
                        else:
                            raise ValueError(f"Agregação não permitida em top_n: {agg}")
                        valor = [{"grupo": _to_scalar(k), "valor": _to_scalar(round(v, 6))} for k, v in serie.items()]

                elif tipo == "timeseries":
                    col_data = _resolver_col_data(trabalho, preferida=item.get("coluna_data"))
                    if not col_data:
                        raise ValueError("timeseries exige coluna de data")
                    freq = (item.get("frequencia") or "ME").upper()
                    agg = (item.get("agregacao") or "count").lower()
                    dt = pd.to_datetime(trabalho[col_data], errors="coerce")
                    df_aux = trabalho.copy()
                    df_aux["__dt__"] = dt
                    df_aux = df_aux[df_aux["__dt__"].notna()]
                    df_aux = df_aux.set_index("__dt__")

                    if agg == "count":
                        serie = df_aux.resample(freq).size()
                    else:
                        if not coluna_valor or coluna_valor not in df_aux.columns:
                            raise ValueError("timeseries com agregação numérica exige coluna_valor")
                        s = pd.to_numeric(df_aux[coluna_valor], errors="coerce")
                        if agg == "sum":
                            serie = s.resample(freq).sum()
                        elif agg == "mean":
                            serie = s.resample(freq).mean()
                        else:
                            raise ValueError(f"Agregação não permitida em timeseries: {agg}")

                    valor = [
                        {"periodo": str(k.date()) if hasattr(k, "date") else str(k), "valor": _to_scalar(round(v, 6) if isinstance(v, (int, float)) else v)}
                        for k, v in serie.items()
                    ]

                else:
                    raise ValueError(f"Tipo não suportado: {tipo}")

                metricas.append({
                    "metric_id": metric_id,
                    "descricao": desc,
                    "tipo": tipo,
                    "status": "ok",
                    "resultado": valor,
                })

            except Exception as exc:
                erro = f"{metric_id}: {exc}"
                erros.append(erro)
                metricas.append({
                    "metric_id": metric_id,
                    "descricao": desc,
                    "tipo": tipo,
                    "status": "erro",
                    "erro": str(exc),
                })

        resultado_obj = {
            "schema_version": "1.0",
            "metricas": metricas,
            "erros": erros,
            "resumo_execucao": {
                "metricas_sucesso": len([m for m in metricas if m.get("status") == "ok"]),
                "metricas_erro": len([m for m in metricas if m.get("status") == "erro"]),
            },
        }
        return {
            "sucesso": len(metricas) > 0 and len([m for m in metricas if m.get("status") == "ok"]) > 0,
            "resultado_texto": json.dumps(resultado_obj, ensure_ascii=False),
            "resultado_obj": resultado_obj,
        }

    if verbose:
        print("[MAESTRO] Iniciando fluxo. Pergunta:", (pergunta[:80] + "...") if len(pergunta) > 80 else pergunta)

    if verbose:
        print("[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...")
    skill_maestro = get_skill_by_id(skills, "maestro")
    model_maestro = skill_maestro.get("model") or model
    msg_analise = (
        pergunta
        + "\n\nRetorne um JSON com uma única chave 'analise' (objeto) contendo: "
        "pergunta (string), dominios_identificados (lista de strings), tipo_resposta_esperada (uma de: factual, analítica, técnica, criativa, comparativa), "
        "complexidade (uma de: baixa, média, alta), informacao_central (string)."
    )
    resp_analise = client.chat.completions.create(
        model=model_maestro,
        messages=[
            {"role": "system", "content": skill_maestro["content"]},
            {"role": "user", "content": msg_analise},
        ],
        response_format={"type": "json_object"},
    )
    raw_analise = (resp_analise.choices[0].message.content or "").strip()
    analise_data = extrair_json(raw_analise)
    analise = (analise_data.get("analise") or analise_data) if isinstance(analise_data, dict) else {}
    if not analise:
        analise = {
            "pergunta": pergunta,
            "dominios_identificados": [],
            "tipo_resposta_esperada": "analítica",
            "complexidade": "média",
            "informacao_central": pergunta,
        }
    contexto_maestro = (
        f"Domínios: {analise.get('dominios_identificados', [])}; "
        f"Tipo: {analise.get('tipo_resposta_esperada', '')}; "
        f"Complexidade: {analise.get('complexidade', '')}; "
        f"Central: {analise.get('informacao_central', '')}"
    )

    if mysql_tabelas or mysql_tabela:
        if verbose:
            print("[MAESTRO] Pré-carregando dados MySQL...")
        resultado_mysql, df_contexto = _preparar_contexto_mysql()
        df_variavel = df_contexto.get("df_variavel")
        if verbose:
            print(f"[MAESTRO] DataFrame disponível em '{df_variavel}'.")
        tabela_ref = (mysql_tabelas[0]["tabela"] if mysql_tabelas else mysql_tabela)
        contexto_maestro = f"{contexto_maestro}; Tabela: {tabela_ref}; DF: {df_variavel}"

    def _invocar_um_agente(skill_id: str) -> Dict:
        """Invocação de um único agente (modo conhecimento ou 2 fases). Retorna o dict de resposta."""
        usa_dataframe = bool(df_contexto and skill_id in _agentes_dataframe)
        if verbose:
            modo = "2 fases" if usa_dataframe else "conhecimento"
            print(f"[MAESTRO] Passo 3: Invocando agente: {skill_id} ({modo})")
        payload_base = {
            "skill_invocada": skill_id,
            "pergunta": pergunta,
            "contexto_maestro": contexto_maestro,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
        }
        if not usa_dataframe:
            payload_base["instrucao"] = "Responda estritamente dentro do seu domínio. Calcule seus scores."
            raw = invocar_agente_maestro(client, skill_id, payload_base, model=model, skills_list=skills)
            parsed = extrair_json(raw)
            if parsed and isinstance(parsed, dict):
                parsed.setdefault("agente_id", skill_id)
                return parsed
            return {
                "agente_id": skill_id,
                "agente_nome": skill_id,
                "pode_responder": False,
                "justificativa_viabilidade": "Resposta não veio em JSON válido.",
                "resposta": "",
                "scores": {},
            }
        payload_fase1 = {**payload_base}
        payload_fase1.update(df_contexto)
        payload_fase1["fase"] = "extracao"
        payload_fase1["instrucao"] = (
            "FASE 1 — EXTRAÇÃO ESTRUTURADA: opere em MODO DATAFRAME. "
            "NÃO gere código Python. Em vez disso, retorne 'perguntas_dados' (lista JSON) seguindo o contrato informado em 'contrato_perguntas_dados'. "
            "Cada pergunta deve ser agregada (sem linhas cruas) e alinhada ao seu domínio."
        )
        if verbose:
            print(f"[MAESTRO]   [{skill_id}] FASE 1: solicitando perguntas_dados...")
        raw1 = invocar_agente_maestro(client, skill_id, payload_fase1, model=model, skills_list=skills)
        parsed1 = extrair_json(raw1)
        if not parsed1 or not isinstance(parsed1, dict):
            return {
                "agente_id": skill_id,
                "agente_nome": skill_id,
                "pode_responder": False,
                "justificativa_viabilidade": "FASE 1 não retornou JSON válido.",
                "resposta": "",
                "scores": {},
            }
        parsed1.setdefault("agente_id", skill_id)
        if not parsed1.get("perguntas_dados"):
            parsed1["pode_responder"] = False
            parsed1["justificativa_viabilidade"] = (
                parsed1.get("justificativa_viabilidade")
                or "FASE 1 não retornou perguntas_dados."
            )
            return parsed1
        exec_info = _executar_perguntas_agregadas(parsed1)
        resultado_extracao = exec_info.get("resultado_obj", {})
        if verbose:
            status_exec = "✅" if exec_info.get("sucesso") else "❌"
            print(f"[MAESTRO]   [{skill_id}] FASE 1 execução {status_exec}: métricas="
                  f"{resultado_extracao.get('resumo_execucao', {}).get('metricas_sucesso', 0)}")
        if not exec_info.get("sucesso"):
            parsed1["resposta"] = (
                (parsed1.get("resposta") or "")
                + "\n\nFalha ao executar perguntas agregadas de FASE 1."
            ).strip()
            parsed1["resultado_execucao"] = resultado_extracao
            return parsed1
        if verbose:
            print(f"[MAESTRO]   [{skill_id}] FASE 2: enviando agregados para interpretação...")
        payload_fase2 = {
            "skill_invocada": skill_id,
            "pergunta": pergunta,
            "contexto_maestro": contexto_maestro,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "fase": "interpretacao",
            "resultado_extracao": resultado_extracao,
            "instrucao": (
                "FASE 2 — INTERPRETAÇÃO: você recebeu em 'resultado_extracao' apenas métricas agregadas reais. "
                "Interprete conforme seu domínio e responda a pergunta do usuário. "
                "Não solicitar nem inferir dados linha a linha. "
                "Retorne o JSON completo de resposta (sem codigo_pandas)."
            ),
        }
        raw2 = invocar_agente_maestro(client, skill_id, payload_fase2, model=model, skills_list=skills)
        parsed2 = extrair_json(raw2)
        if parsed2 and isinstance(parsed2, dict):
            parsed2.setdefault("agente_id", skill_id)
            parsed2["resultado_execucao"] = resultado_extracao
            return parsed2
        parsed1["resultado_execucao"] = resultado_extracao
        parsed1["resposta"] = (
            (parsed1.get("resposta") or "")
            + "\n\nFASE 2 indisponível; retornando somente resultados agregados executados."
        ).strip()
        return parsed1

    with ThreadPoolExecutor(max_workers=len(agentes)) as executor:
        futures = []
        for i, sid in enumerate(agentes):
            if i > 0:
                time.sleep(time_interval_agents)
            futures.append(executor.submit(_invocar_um_agente, sid))
        respostas_agentes = [f.result() for f in futures]

    para_avaliador = [r for r in respostas_agentes if r.get("pode_responder") is True]
    if verbose:
        print("[MAESTRO] Passo 4: Respostas a enviar ao avaliador:", len(para_avaliador), "de", len(respostas_agentes))

    avaliacao = None
    if not para_avaliador:
        entrega_final = (
            "## Resposta do Maestro\n\n**Pergunta:** " + pergunta
            + "\n\nNenhum agente qualificado para responder. "
            "Sugestão: reformule a pergunta ou consulte um domínio mais específico."
        )
    else:
        if verbose:
            print("[MAESTRO] Passo 5: Invocando avaliador de coerência...")
        payload_aval = {
            "pergunta_original": pergunta,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "respostas_coletadas": [
                {
                    "agente_id": r.get("agente_id"),
                    "agente_nome": r.get("agente_nome", r.get("agente_id")),
                    "resposta": r.get("resposta", ""),
                    "scores_agente": r.get("scores") or {},
                    "limitacoes_da_resposta": r.get("limitacoes_da_resposta", ""),
                }
                for r in para_avaliador
            ],
        }
        skill_aval = get_skill_by_id(skills, "avaliador-coerencia")
        model_aval = skill_aval.get("model") or model
        user_aval = (
            json.dumps(payload_aval, ensure_ascii=False)
            + "\n\nRetorne APENAS um JSON com: avaliacao_completa (lista de objetos com "
            "agente_id, agente_nome, scores_avaliador, score_total, status, observacoes), "
            "ranking_final (lista de agente_id), conflitos_detectados, respostas_excluidas, threshold_utilizado."
        )
        resp_aval = client.chat.completions.create(
            model=model_aval,
            messages=[
                {"role": "system", "content": skill_aval["content"]},
                {"role": "user", "content": user_aval},
            ],
            response_format={"type": "json_object"},
        )
        raw_aval = (resp_aval.choices[0].message.content or "").strip()
        avaliacao = extrair_json(raw_aval)

        if verbose:
            ranking = (avaliacao or {}).get("ranking_final", [])
            print("[MAESTRO] Avaliador retornou ranking:", ranking)

        if verbose:
            print("[MAESTRO] Passo 6: Formatando entrega ao usuário.")
        entrega_final = _formatar_entrega(pergunta, respostas_agentes, avaliacao, para_avaliador)

    if verbose:
        print("[MAESTRO] Fluxo concluído.")
    return {
        "analise": analise,
        "respostas_agentes": respostas_agentes,
        "avaliacao": avaliacao,
        "entrega_final": entrega_final,
        "resultado_mysql": resultado_mysql,
        "df_variavel": df_variavel,
    }


In [8]:
# PERGUNTA_TESTE = (
#     "Quais são as melhores práticas para uma startup de SaaS precificar o produto "
#     "e quais ferramentas técnicas (APIs, billing) ajudam a implementar isso?"
# )

# resultado = executar_fluxo_maestro(client=client, pergunta=PERGUNTA_TESTE, model=model, agentes=["agente-tecnico", "agente-financeiro"], verbose=True)
# print(resultado["entrega_final"])

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA ÚNICA — Pergunta ao Maestro e ele busca 10 linhas da tabela
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. Pergunta em linguagem natural ─────────────────────────────────────────

# PERGUNTA = "Me mostre 10 linhas da tabela de servicos"

# # ── 2. Maestro identifica qual tabela buscar ──────────────────────────────────

# skill_maestro = get_skill_by_id(skills, "maestro")

# resp = client.chat.completions.create(
#     model=model,
#     messages=[
#         {"role": "system", "content": skill_maestro["content"]},
#         {"role": "user",   "content": (
#             f"{PERGUNTA}\n\n"
#             "Retorne APENAS um JSON com a chave 'tabela' contendo o nome da tabela "
#             "que o usuário quer consultar. Ex: {\"tabela\": \"vendas\"}"
#         )},
#     ],
#     response_format={"type": "json_object"},
# )

# tabela = extrair_json(resp.choices[0].message.content).get("tabela", "")
# print(f"[MAESTRO] Tabela identificada: '{tabela}'")

# # ── 3. agente-mysql busca as 10 linhas ───────────────────────────────────────

# agent = MySQLAgent(
#     host    = os.environ.get("MYSQL_HOST"),
#     banco   = os.environ.get("MYSQL_DATABASE"),
#     usuario = os.environ.get("MYSQL_USER"),
#     senha   = os.environ.get("MYSQL_PASSWORD"),
# )

# resultado = agent.carregar_tabela(tabela, limite=10, verbose=False)

# if not resultado["sucesso"]:
#     print(f"[ERRO] {resultado['erro']}")
# else:
#     df = resultado["dataframe"]
#     print(f"\n✅ Tabela '{tabela}' — 10 primeiras linhas:\n")
#     print(df.to_string(index=False))

In [10]:
# # ─────────────────────────────────────────────────────────────────────────────
# # CÉLULA: MySQL → agente-dados → executa código Pandas
# # ─────────────────────────────────────────────────────────────────────────────

# # ── Configuração ──────────────────────────────────────────────────────────────

# PERGUNTA   = "Quais os 5 serviços com maior custo_fixo e estão ativos?"
# TABELA     = "servicos"

# resultado = executar_fluxo_maestro(
#     client=client,
#     pergunta=PERGUNTA,
#     model=model,
#     agentes=["agente-dados"],
#     verbose=True,
#     mysql_host=os.environ.get("MYSQL_HOST"),
#     mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
#     mysql_usuario=os.environ.get("MYSQL_USER"),
#     mysql_senha=os.environ.get("MYSQL_PASSWORD"),
#     mysql_banco=os.environ.get("MYSQL_DATABASE"),
#     mysql_tabela=TABELA,
#     mysql_limite=50.000,
#     mysql_filtro_where="",
#     mysql_injetar_namespace=globals(),
# )

# resp = next((r for r in resultado["respostas_agentes"] if r.get("agente_id") == "agente-dados"), {})

# print(f"\n[agente-dados] pode_responder: {resp.get('pode_responder')}")
# print(f"[agente-dados] score_final:    {resp.get('scores', {}).get('score_final')}")
# print(f"[agente-dados] {resp.get('justificativa_viabilidade')}")

# # ── Executa o código Pandas gerado ───────────────────────────────────────────

# codigo = resp.get("codigo_pandas", "")

# if not codigo:
#     print("\n[AVISO] agente-dados não retornou codigo_pandas.")
#     print("Resposta:", resp.get("resposta"))
# else:
#     print(f"\n{'─'*60}")
#     print("CÓDIGO GERADO PELO agente-dados:")
#     print('─'*60)
#     print(codigo)
#     print('─'*60)
#     print("\nRESULTADO:\n")
#     exec(codigo)   # executa no namespace atual — df_servicos já está disponível

In [11]:
PERGUNTA_TESTE = (
    "quais concessionárias vendem mais?"
)

resultado = executar_fluxo_maestro(
    client=client,
    pergunta=PERGUNTA_TESTE,
    model=model,
    agentes=["agente-dados", "agente-negocios"],
    verbose=True,
    mysql_host=os.environ.get("MYSQL_HOST"),
    mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
    mysql_usuario=os.environ.get("MYSQL_USER"),
    mysql_senha=os.environ.get("MYSQL_PASSWORD"),
    mysql_banco=os.environ.get("MYSQL_DATABASE"),
    # mysql_tabela="os_servicos",
    mysql_tabelas=[
        {"tabela": "os", "alias": "os"},
        {"tabela": "os_servicos", "alias": "oss", "fk": "os.id = oss.os_id"},
        {"tabela": "servicos", "alias": "ser", "fk": "oss.servico_id = ser.id", "colunas": ["ser.nome AS servico_nome"]},
        {"tabela": "concessionarias", "alias": "con", "fk": "os.concessionaria_id = con.id", "colunas": ["con.nome AS concessionaria_nome"]}
    ],
    mysql_limite=10000,
    mysql_filtro_where="os.deleted_at IS NULL",
    mysql_injetar_namespace=globals(),
)
print(resultado["entrega_final"])

[MAESTRO] Iniciando fluxo. Pergunta: quais concessionárias vendem mais?
[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...


2026-03-16 17:53:58,730 [openai._base_client] DEBUG: Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-c3c6f479-564b-4b4a-89b5-13aa46357232', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': '---\nname: maestro\nmodel: gpt-5-mini\ndescription: >\n  Orquestrador central de múltiplos agentes especializados. Ative esta skill quando a pergunta do usuário\n  envolver mais de um domínio de conhecimento, ou quando precisar de respostas multidisciplinares com alta\n  precisão. O Maestro analisa a pergunta, seleciona os agentes mais adequados do registro de skills\n  disponíveis, coleta respostas com scores, envia ao Avaliador de Coerência e entrega ao usuário apenas\n  as respostas mais relevantes e consistentes. Use sempre que o usuário mencionar "agentes", "múltiplas\n  perspectivas", "orquestração", ou quando a pergunta cruzar domínios como finanças+tecnologia,\n  saúde+jurídico, dados+negócios,

[MAESTRO] Pré-carregando dados MySQL...
[agente-mysql] Conectando ao banco 'smart'...
[agente-mysql] ✅ 4 tabela(s) validada(s): ['os', 'os_servicos', 'servicos', 'concessionarias']
[agente-mysql] Linhas na tabela principal 'os': 173,049
[agente-mysql] Query gerada:
SELECT `os`.*, `oss`.*, ser.nome AS servico_nome, con.nome AS concessionaria_nome
FROM `os` AS `os`
  LEFT JOIN `os_servicos` AS `oss` ON os.id = oss.os_id
  LEFT JOIN `servicos` AS `ser` ON oss.servico_id = ser.id
  LEFT JOIN `concessionarias` AS `con` ON os.concessionaria_id = con.id
WHERE os.deleted_at IS NULL
ORDER BY `os`.id DESC
LIMIT 10000

[agente-mysql] Executando query (limite=10,000)...


/tmp/ipykernel_738602/1922295981.py:139: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    1
Name: email_garantia_enviado, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  amostra.iloc[:, i] = amostra.iloc[:, i].astype(object)
/tmp/ipykernel_738602/1922295981.py:149: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0    332946.0
1    332945.0
2    332944.0
3    332942.0
4    332943.0
5    332941.0
6    332940.0
7    332932.0
Name: id, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  amostra.iloc[:, i] = amostra.iloc[:, i].astype(object)
2026-03-16 17:54:13,526 [openai._base_client] DEBUG: Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'st

[agente-mysql] ✅ DataFrame carregado: 10,000 linhas × 115 colunas → 'df_os'
[agente-mysql] ⚠️  Parcial: 10,000 de 173,049
[agente-mysql] ✅ 'df_os' injetado com 10,000 linhas.
[MAESTRO] DataFrame disponível em 'df_os'.
[MAESTRO] Passo 3: Invocando agente: agente-dados (2 fases)
[MAESTRO]   [agente-dados] FASE 1: solicitando perguntas_dados...


...

[MAESTRO]   [agente-negocios] FASE 1 execução ❌: métricas=0


2026-03-16 17:55:56,340 [httpcore.http11] DEBUG: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Mon, 16 Mar 2026 20:55:56 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9dd6a89929a1ca5e-GIG'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'carsoul-pliqq2'), (b'openai-processing-ms', b'31782'), (b'openai-project', b'proj_hxilwBbr6yeydV4YtYGbjIqZ'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3996328'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-

[MAESTRO] Passo 4: Respostas a enviar ao avaliador: 2 de 2
[MAESTRO] Passo 5: Invocando avaliador de coerência...


2026-03-16 17:56:23,233 [httpcore.http11] DEBUG: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Mon, 16 Mar 2026 20:56:23 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9dd6a9618aa3ca5e-GIG'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'carsoul-pliqq2'), (b'openai-processing-ms', b'26561'), (b'openai-project', b'proj_hxilwBbr6yeydV4YtYGbjIqZ'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3997015'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-

[MAESTRO] Avaliador retornou ranking: ['agente-dados', 'agente-negocios']
[MAESTRO] Passo 6: Formatando entrega ao usuário.
[MAESTRO] Fluxo concluído.
## Resposta do Maestro

**Pergunta:** quais concessionárias vendem mais?
**Agentes consultados:** 2 | **Respostas qualificadas:** 2
---
### Analista de Dados — agente-dados
*Score de Confiança: 87%*

Resumo executivo e interpretação técnica:
1) Rankings observados
- Por volume (unidades): Top 3 concessionárias são os IDs 23 (545 OS), 33 (543 OS) e 105 (506 OS). Estes representam os maiores volumes de serviços/OS no período analisado (período não informado nas métricas entregues — necessário para contexto temporal). 
- Por receita (soma de valor_venda_real): Top 3 são IDs 68 (371.239,00), 34 (338.221,00) e 105 (307.917,10). Note que o ranking por receita difere do ranking por volume: por exemplo, ID 68 é o primeiro por receita mas não aparece no top-10 por volume. Isso indica que alguns concessionários geram ticket médio bem maior (mais r